In [58]:
import requests
import json
from pathlib import Path
import pandas as pd
from itertools import combinations
import networkx as nx
!pip install GitPython unidecode

In [2]:
api_key = "irjTGQKkPBoFg4bRlinSRu"

In [3]:
year = 2024 # for now it's just 2024 then i'll do other years
work_type = "article"
Path("data").mkdir(exist_ok=True)

In [4]:
filter_string = (
    f"publication_year:{year},"
    f"institutions.country_code:us|gb|ru|it|br|kr,"  # US, UK, Russia, Italy, Brazil, South Korea 
    f"primary_topic.field.id:20,"
    f"type:{work_type},"
    f"is_paratext:false")

print(filter_string)

publication_year:2024,institutions.country_code:us|gb|ru|it|br|kr,primary_topic.field.id:20,type:article,is_paratext:false


In [5]:
url = "https://api.openalex.org/works"

cursor = "*"
all_works = []

while True:
    params = {
        "filter": filter_string,
        "per_page": 100,
        "cursor": cursor,
        "api_key": api_key}

    r = requests.get(url, params=params)
    data = r.json()

    results = data["results"]
    all_works.extend(results)

    cursor = data["meta"]["next_cursor"]

    if not results or cursor is None:
        break

print("downloaded works num:", len(all_works))

downloaded works num: 33501


In [6]:
countries = {"us", "gb", "ru", "it", "br", "kr"}

In [7]:
rows = []

for work in all_works:
    paper_id = work.get("id")
    paper_title = work.get("display_name")
    publication_year = work.get("publication_year")
    
    for authorship in work.get("authorships", []):
        author = authorship.get("author", {})
        author_id = author.get("id")
        author_name = author.get("display_name")
        
        institutions = authorship.get("institutions", [])
        
        author_countries = []
        for inst in institutions:
            country_code = inst.get("country_code")
            
            if country_code is not None:
                country_code = country_code.lower()
                
                if country_code in countries and country_code not in author_countries:
                    author_countries.append(country_code)
        
        if len(author_countries) == 0:
            continue
        if author_id is None:
            continue
        
        rows.append({
            "paper_id": paper_id,
            "paper_title": paper_title,
            "publication_year": publication_year,
            "author_id": author_id,
            "author_name": author_name,
            "author_countries": ";".join(author_countries)})

In [9]:
authors_df = pd.DataFrame(rows)
authors_df.head()

,paper_id,paper_title,publication_year,author_id,author_name,author_countries
0,https://openalex.org/W4399848789,GPTs are GPTs: Labor market impact potential o...,2024,https://openalex.org/A5065917408,Tyna Eloundou,us
1,https://openalex.org/W4399848789,GPTs are GPTs: Labor market impact potential o...,2024,https://openalex.org/A5028772381,Pamela Mishkin,us
2,https://openalex.org/W4399848789,GPTs are GPTs: Labor market impact potential o...,2024,https://openalex.org/A5018502975,Daniel L. Rock,us
3,https://openalex.org/W4390954288,Corporate Climate Risk: Measurements and Respo...,2024,https://openalex.org/A5100404077,Qing Li,us
4,https://openalex.org/W4390954288,Corporate Climate Risk: Measurements and Respo...,2024,https://openalex.org/A5005717316,Hongyu Shan,us


In [10]:
authors_df.sample(10)

,paper_id,paper_title,publication_year,author_id,author_name,author_countries
21303,https://openalex.org/W4391541271,How Socio-economic Inequalities Cluster People...,2024,https://openalex.org/A5070943287,Lance A. Waller,us
4481,https://openalex.org/W4392975721,Identifying key products to trigger new export...,2024,https://openalex.org/A5043942237,Giambattista Albora,it
75303,https://openalex.org/W4405748765,P55 Navigating Change: A Comparative Analysis ...,2024,https://openalex.org/A5010380744,Ciprian-Paul Radu,gb
46777,https://openalex.org/W4392917582,The founding of the federal banking system as ...,2024,https://openalex.org/A5004082658,Charles W. Calomiris,us
15908,https://openalex.org/W4402468485,Asymmetry and non-linearity in exchange rate p...,2024,https://openalex.org/A5086269486,In Kyung Kim,kr
34110,https://openalex.org/W4402517385,Family caregivers’ perceptions of the quality ...,2024,https://openalex.org/A5061635395,Fernando Rodrigues Peixoto Quaresma,br
34486,https://openalex.org/W4403061946,Life-Threatening Reaction to Lifesaving Medica...,2024,https://openalex.org/A5099471707,Tia Bimal,us
36245,https://openalex.org/W4405335340,Long Dosing Intervals of Parenteral Antiosteop...,2024,https://openalex.org/A5087531145,Olivier Q. Groot,us
78342,https://openalex.org/W4407987977,How can the tragedy of the commons be prevente...,2024,https://openalex.org/A5116435929,Gökçe Dayanı,us
65089,https://openalex.org/W4402769415,"Japan: Nippon Credit Bank Capital Injection, 1997",2024,https://openalex.org/A5107510044,Owen Heaphy,us


In [11]:
authors_df.to_csv("data/authors_2024.csv", index = False)

In [20]:
pair_all = []

for paper_id, group in authors_df.groupby("paper_id"):
    group = group.drop_duplicates(subset = "author_id").copy()
    group = group.sort_values("author_id")
    
    authors = group[["author_id", "author_name", "author_countries"]].to_dict("records")
    
    if len(authors) < 2:
        continue
    
    for a1, a2 in combinations(authors, 2):
        pair_all.append({
            "paper_id": paper_id,
            "author_1_id": a1["author_id"],
            "author_1_name": a1["author_name"],
            "author_1_countries": a1["author_countries"],
            "author_2_id": a2["author_id"],
            "author_2_name": a2["author_name"],
            "author_2_countries": a2["author_countries"]})

In [21]:
pairs_df = pd.DataFrame(pair_all)
pairs_df.head()

,paper_id,author_1_id,author_1_name,author_1_countries,author_2_id,author_2_name,author_2_countries
0,https://openalex.org/W1502437201,https://openalex.org/A5039986567,Stephen Hunt,gb,https://openalex.org/A5113617222,William Bygrave,us
1,https://openalex.org/W1518857748,https://openalex.org/A5050178053,James E. Rauch,us,https://openalex.org/A5101629348,Ying Feng,us
2,https://openalex.org/W1585540642,https://openalex.org/A5003626336,Paolo Maria Russo,it,https://openalex.org/A5039266573,Aniello Visciano,it
3,https://openalex.org/W1585540642,https://openalex.org/A5003626336,Paolo Maria Russo,it,https://openalex.org/A5049995257,Daniela Ugliano,it
4,https://openalex.org/W1585540642,https://openalex.org/A5003626336,Paolo Maria Russo,it,https://openalex.org/A5104145854,Giorgio Liguori,it


In [22]:
print(len(pairs_df))
print(pairs_df["paper_id"].nunique())

197656
17470


In [23]:
pairs_df.to_csv("data/pairs_2024.csv", index = False)
print("saved")

saved


In [24]:
edges_df = (
    pairs_df.groupby(["author_1_id", "author_1_name", "author_1_countries",
                      "author_2_id", "author_2_name", "author_2_countries"], 
                     as_index=False).size().rename(columns={"size": "weight"}))

In [25]:
edges_df.head()

,author_1_id,author_1_name,author_1_countries,author_2_id,author_2_name,author_2_countries,weight
0,https://openalex.org/A5000000076,I. M. Shor,ru,https://openalex.org/A5063275369,Dildarakhon A. Shelestova,ru,1
1,https://openalex.org/A5000005418,Márcia Mascarenhas Alemão,br,https://openalex.org/A5019570644,João Flávio de Freitas Almeida,br,1
2,https://openalex.org/A5000005418,Márcia Mascarenhas Alemão,br,https://openalex.org/A5029958956,Samuel Vieira Conceição,br,1
3,https://openalex.org/A5000007462,Kaitlin Casassa,us,https://openalex.org/A5048869928,Sharvari Karandikar,us,1
4,https://openalex.org/A5000007462,Kaitlin Casassa,us,https://openalex.org/A5087366708,Rochelle L. Dalla,us,1


In [26]:
print(len(edges_df))
print(edges_df["weight"].sum())
print(edges_df["weight"].max())
print((edges_df["weight"] == 1).sum())

185296
197656
35
176686


In [27]:
different_edges = edges_df[edges_df["author_1_countries"] != edges_df["author_2_countries"]].copy()

print("edges between dif countries:", len(different_edges))

edges between dif countries: 13761


In [29]:
authors_df = authors_df.drop_duplicates(subset=["paper_id", "author_id"]).copy()

In [31]:
node_rows = []

for author_id, group in authors_df.groupby("author_id"):
    author_name = group["author_name"].iloc[0]
    
    country_counts = {}
    
    for x in group["author_countries"]:
        for c in x.split(";"):
            country_counts[c] = country_counts.get(c, 0) + 1
    
    all_countries = sorted(country_counts.keys())
    main_country = max(country_counts, key=country_counts.get)
    num_papers = group["paper_id"].nunique()
    
    node_rows.append({
        "author_id": author_id,
        "author_name": author_name,
        "author_countries": ";".join(all_countries),
        "main_country": main_country,
        "num_papers": num_papers})

nodes_df = pd.DataFrame(node_rows)
nodes_df.head()

,author_id,author_name,author_countries,main_country,num_papers
0,https://openalex.org/A5000000076,I. M. Shor,ru,ru,1
1,https://openalex.org/A5000005418,Márcia Mascarenhas Alemão,br,br,1
2,https://openalex.org/A5000007462,Kaitlin Casassa,us,us,1
3,https://openalex.org/A5000011831,Jesse T. Wright,us,us,1
4,https://openalex.org/A5000013148,Maksim А. Nikulin,ru,ru,1


In [33]:
print("n of nodes:", len(nodes_df))

n of nodes: 64664


In [34]:
nodes_df["main_country"].value_counts()

main_country
us    36916
gb    10749
ru     5071
it     4983
br     4890
kr     2055
Name: count, dtype: int64

In [35]:
nodes_df[nodes_df["author_countries"].str.contains(";")].head(10)

,author_id,author_name,author_countries,main_country,num_papers
54,https://openalex.org/A5000113430,Dimitri Vayanos,gb;us,gb,2
190,https://openalex.org/A5000368607,S. Bello,gb;us,gb,1
267,https://openalex.org/A5000534570,Juan de Díos Tena,gb;it,gb,1
327,https://openalex.org/A5000628175,Mohammed Benidris,gb;it;us,us,1
337,https://openalex.org/A5000642577,Rose‐Mary Sargent,gb;us,gb,1
343,https://openalex.org/A5000650303,S. Alex Yang,gb;us,gb,2
466,https://openalex.org/A5000892248,Francesco Saverio Mennini,gb;it,it,1
602,https://openalex.org/A5001187199,Benedetta Armocida,it;us,it,3
625,https://openalex.org/A5001227208,Florencia López Bóo,gb;us,us,1
649,https://openalex.org/A5001285815,Matthias Neuber,gb;us,us,1


In [37]:
nodes_df.to_csv("data/nodes_2024.csv", index=False)
print("saved")

saved


In [39]:
g = nx.Graph()

for _, row in edges_df.iterrows():
    g.add_edge(row["author_1_id"], row["author_2_id"], weight=row["weight"])

print("nodes:", g.number_of_nodes())
print("edges:", g.number_of_edges())

nodes: 52657
edges: 184696


In [40]:
components = list(nx.connected_components(g))

print("len(components)", len(components))
print("len(max(components))", len(max(components, key=len)))

len(components) 10457
len(max(components)) 8200


In [41]:
degree_dict = dict(g.degree())
weighted_degree_dict = dict(g.degree(weight="weight"))

In [42]:
nodes_df["degree"] = nodes_df["author_id"].map(degree_dict).fillna(0).astype(int)
nodes_df["weighted_degree"] = nodes_df["author_id"].map(weighted_degree_dict).fillna(0).astype(int)

nodes_df.head()

,author_id,author_name,author_countries,main_country,num_papers,degree,weighted_degree
0,https://openalex.org/A5000000076,I. M. Shor,ru,ru,1,1,1
1,https://openalex.org/A5000005418,Márcia Mascarenhas Alemão,br,br,1,2,2
2,https://openalex.org/A5000007462,Kaitlin Casassa,us,us,1,2,2
3,https://openalex.org/A5000011831,Jesse T. Wright,us,us,1,3,3
4,https://openalex.org/A5000013148,Maksim А. Nikulin,ru,ru,1,0,0


In [43]:
largest_component = max(nx.connected_components(g), key=len)
largest_component_set = set(largest_component)

nodes_df["is_in_giant_component"] = nodes_df["author_id"].isin(largest_component_set)

In [44]:
print(nodes_df[["num_papers", "degree", "weighted_degree"]].describe())
nodes_df[nodes_df["degree"] == 204]

         num_papers        degree  weighted_degree
count  64664.000000  64664.000000     64664.000000
mean       1.240721      5.712483         6.090591
std        0.872622     12.275368        13.566763
min        1.000000      0.000000         0.000000
25%        1.000000      1.000000         1.000000
50%        1.000000      3.000000         3.000000
75%        1.000000      6.000000         6.000000
max       36.000000    204.000000       210.000000


In [45]:
nodes_df["is_in_giant_component"].value_counts()

is_in_giant_component
False    56464
True      8200
Name: count, dtype: int64

In [46]:
nodes_df.to_csv("data/nodes_2024_with_metrics.csv", index=False)
print("saved")

saved


In [47]:
nodes_df.groupby("main_country")[["num_papers", "degree", "weighted_degree"]].mean()

,num_papers,degree,weighted_degree
main_country,,,
br,1.140695,3.266258,3.392025
gb,1.248767,5.574193,5.814773
it,1.304034,5.001204,5.232791
kr,1.274453,2.502190,2.614599
ru,1.170972,1.862552,1.943009
us,1.250786,6.880350,7.407384


In [54]:
nodes_df.groupby("main_country")[["num_papers", "degree", "weighted_degree"]].agg(["mean", "median"])

num_papers           degree        weighted_degree       
                   mean median      mean median            mean median
main_country                                                          
br             1.140695    1.0  3.266258    2.0        3.392025    2.0
gb             1.248767    1.0  5.574193    2.0        5.814773    2.0
it             1.304034    1.0  5.001204    3.0        5.232791    3.0
kr             1.274453    1.0  2.502190    2.0        2.614599    2.0
ru             1.170972    1.0  1.862552    1.0        1.943009    1.0
us             1.250786    1.0  6.880350    3.0        7.407384    3.0

In [55]:
nodes_df.groupby("main_country")["is_in_giant_component"].mean()

main_country
br    0.003681
gb    0.118988
it    0.065824
kr    0.011192
ru    0.000197
us    0.177457
Name: is_in_giant_component, dtype: float64

In [51]:
print((nodes_df["degree"] == 0).sum())

12007


The following code cells as well as this method of extracting gender from name are taken from https://github.com/IES-platform/r4r_gender/tree/main

In [61]:
import requests
import zipfile
from io import BytesIO
import os
import sys

url = "https://github.com/ClemSternWIPO/gender_it/archive/refs/heads/main.zip"

r = requests.get(url)
r.raise_for_status()

z = zipfile.ZipFile(BytesIO(r.content))
z.extractall("gender_it_local")

print(os.listdir("gender_it_local"))

['gender_it-main']


In [62]:
sys.path.append("gender_it_local/gender_it-main")
import gender_it_functions as gf

In [64]:
import re
import numpy as np

def extract_first_name(full_name):
    if pd.isna(full_name):
        return np.nan
    
    name = str(full_name).strip()
    name = re.sub(r"\s+", " ", name)
    
    if name == "":
        return np.nan
    
    first = name.split(" ")[0].strip()
    first_clean = first.replace(".", "")
    
    if len(first_clean) <= 1:
        return np.nan
    
    if not re.search(r"[A-Za-zÀ-ÿĀ-žА-Яа-я]", first_clean):
        return np.nan
    
    return first_clean

In [65]:
gender_input["first_name"] = gender_input["name"].apply(extract_first_name)
gender_input.head()

,author_id,name,country_code,first_name
0,https://openalex.org/A5000000076,I. M. Shor,ru,NaN
1,https://openalex.org/A5000005418,Márcia Mascarenhas Alemão,br,Márcia
2,https://openalex.org/A5000007462,Kaitlin Casassa,us,Kaitlin
3,https://openalex.org/A5000011831,Jesse T. Wright,us,Jesse
4,https://openalex.org/A5000013148,Maksim А. Nikulin,ru,Maksim


In [66]:
print("missing first_name:", gender_input["first_name"].isna().sum())

missing first_name: 4003


In [68]:
gender_input[gender_input["first_name"].isna()]["country_code"].value_counts()

country_code
us    1704
ru    1236
gb     618
it     244
br     125
kr      76
Name: count, dtype: int64

In [73]:
dict_path = "gender_it_local/gender_it-main/dictionaries/"

In [74]:
d2_1 = pd.read_csv(dict_path + "d2_1.csv.gz", compression="gzip")
d2_2 = pd.read_csv(dict_path + "d2_2.csv.gz", compression="gzip")
d2_3 = pd.read_csv(dict_path + "d2_3.csv.gz", compression="gzip")

d2 = pd.concat([d2_1, d2_2, d2_3], ignore_index=True)

d2.to_csv(dict_path + "d2.csv.gz", index=False, compression="gzip")

print("saved:", dict_path + "d2.csv.gz")
print(d2.shape)

saved: gender_it_local/gender_it-main/dictionaries/d2.csv.gz
(26043223, 3)


In [75]:
import os
print(os.listdir(dict_path))

['d1.csv.gz', 'd2.csv.gz', 'd2_1.csv.gz', 'd2_2.csv.gz', 'd2_3.csv.gz', 'd3.csv.gz', 'Data description']


In [76]:
test_df = gender_input.dropna(subset=["first_name"])[["first_name", "country_code"]].copy()
test_df = test_df.rename(columns={"first_name": "name"}).head(20)

test_gendered = gf.get_gender(test_df, name_column="name", country_column="country_code", threshold=0.85, path=dict_path)

print(test_gendered.columns.tolist())
test_gendered.head()

Step 1 - Reading the name-country-gender dictionary
reading the dictionnary.
Step 2 - Reading the name-language-gender dictionary
reading the dictionnary.
Step 3 - Reading the name-gender dictionary
reading the dictionnary.
dff     name_id clean_name  surname_position clean_country_column
19       19        fei                 3                   US
Results distribution is as follows:
            count  Percentage
gender                      
F             10        50.0
M              9        45.0
not found      1         5.0
['level', 'gender', 'F', 'M', 'name', 'country_code']


,level,gender,F,M,name,country_code
0,1,F,1.000000,0.0,Márcia,br
1,1,F,0.998667,0.0,Kaitlin,us
2,2,M,0.000000,1.0,Jesse,us
3,1,M,0.000000,1.0,Maksim,ru
4,1,M,0.000000,1.0,Richard,gb


In [77]:
gender_work = gender_input.dropna(subset=["first_name"])[["author_id", "first_name", "country_code"]].copy()
gender_work = gender_work.rename(columns={"first_name": "name"})

gendered_nodes = gf.get_gender(gender_work, name_column="name", country_column="country_code", threshold=0.85, path=dict_path)

Step 1 - Reading the name-country-gender dictionary
reading the dictionnary.
Step 2 - Reading the name-language-gender dictionary
reading the dictionnary.
Step 3 - Reading the name-gender dictionary
reading the dictionnary.
dff        name_id clean_name  surname_position clean_country_column
27270    27270  huibrecht                 3                   US
Results distribution is as follows:
            count  Percentage
gender                      
M          32046   52.828011
F          21762   35.874780
not found   6583   10.852113
?            270    0.445097


In [78]:
print(gendered_nodes.columns.tolist())
gendered_nodes.head()

['level', 'gender', '?', 'F', 'M', 'author_id', 'name', 'country_code']


,level,gender,?,F,M,author_id,name,country_code
0,1,F,0.0,1.000000,0.0,https://openalex.org/A5000005418,Márcia,br
1,1,F,0.0,0.998667,0.0,https://openalex.org/A5000007462,Kaitlin,us
2,2,M,0.0,0.000000,1.0,https://openalex.org/A5000011831,Jesse,us
3,1,M,0.0,0.000000,1.0,https://openalex.org/A5000013148,Maksim,ru
4,1,M,0.0,0.000000,1.0,https://openalex.org/A5000018947,Richard,gb


In [79]:
gendered_nodes["name_inferred_gender"] = gendered_nodes["gender"].replace({"F": "female", "M": "male", "?": "unknown", "not found": "unknown"})

In [80]:
nodes_df = nodes_df.merge(gendered_nodes[["author_id", "gender", 
                                          "name_inferred_gender", "F", "M", "level"]],
                          on="author_id", how="left")

nodes_df["name_inferred_gender"] = nodes_df["name_inferred_gender"].fillna("unknown")
nodes_df["gender"] = nodes_df["gender"].fillna("not found")

In [81]:
nodes_df["gender"].value_counts(dropna=False)

gender
M            32046
F            21762
not found    10586
?              270
Name: count, dtype: int64

In [82]:
nodes_df["name_inferred_gender"].value_counts(dropna=False)

name_inferred_gender
male       32046
female     21762
unknown    10856
Name: count, dtype: int64

In [83]:
pd.crosstab(nodes_df["level"], nodes_df["name_inferred_gender"], dropna=False)

name_inferred_gender,female,male,unknown
level,,,
1.0,17971,13726,270
2.0,1726,15671,0
3.0,2065,2649,6583
NaN,0,0,4003


In [84]:
nodes_df.to_csv("data/nodes_2024_with_gender.csv", index=False)
print("saved")

saved


In [89]:
pd.crosstab(nodes_df["main_country"], nodes_df["name_inferred_gender"])

name_inferred_gender,female,male,unknown
main_country,,,
br,1587,2680,623
gb,3490,5745,1514
it,1722,2920,341
kr,333,417,1305
ru,1609,1588,1874
us,13021,18696,5199


In [86]:
pd.crosstab(nodes_df["main_country"], nodes_df["name_inferred_gender"], normalize="index")

name_inferred_gender,female,male,unknown
main_country,,,
br,0.324540,0.548057,0.127403
gb,0.324681,0.534468,0.140850
it,0.345575,0.585992,0.068433
kr,0.162044,0.202920,0.635036
ru,0.317294,0.313153,0.369552
us,0.352720,0.506447,0.140833


In [90]:
analysis_df = nodes_df[nodes_df["name_inferred_gender"].isin(["female", "male"])].copy()

analysis_df.groupby("name_inferred_gender")[["num_papers", "degree", "weighted_degree"]].mean()

,num_papers,degree,weighted_degree
name_inferred_gender,,,
female,1.193411,6.780673,7.265647
male,1.290863,5.370187,5.727954


In [91]:
analysis_df.groupby(["main_country", "name_inferred_gender"])[["num_papers", "degree", "weighted_degree"]].mean()

num_papers    degree  weighted_degree
main_country name_inferred_gender                                       
br           female                  1.080025  3.789540         3.893510
             male                    1.184701  2.881343         3.026866
gb           female                  1.201433  6.119198         6.395989
             male                    1.297824  5.342559         5.585205
it           female                  1.219512  5.418118         5.592334
             male                    1.376370  4.485959         4.746918
kr           female                  1.132132  2.438438         2.519520
             male                    1.347722  2.177458         2.287770
ru           female                  1.147296  1.911746         1.968303
             male                    1.222922  1.707179         1.816751
us           female                  1.208893  8.215421         8.906996
             male                    1.295090  6.255884         6.721170

In [93]:
strict_df = nodes_df[(nodes_df["name_inferred_gender"].isin(["female", "male"])) & (nodes_df["level"].isin([1, 2]))].copy()

In [94]:
strict_df.groupby(["main_country", "name_inferred_gender"])[["num_papers", "degree", "weighted_degree"]].mean()

num_papers    degree  weighted_degree
main_country name_inferred_gender                                       
br           female                  1.082260  3.933598         4.051536
             male                    1.175334  2.896457         3.043718
gb           female                  1.199206  6.340972         6.628170
             male                    1.300421  5.454296         5.703426
it           female                  1.222749  5.394550         5.572275
             male                    1.380688  4.472734         4.737409
kr           female                  1.179245  3.094340         3.198113
             male                    1.342466  3.095890         3.226027
ru           female                  1.153318  1.893974         1.955759
             male                    1.207038  1.697218         1.805237
us           female                  1.210154  8.365556         9.062307
             male                    1.298093  6.335598         6.815047

In [100]:
strict_df.groupby(["main_country", "name_inferred_gender"])[["num_papers", "degree", "weighted_degree"]].agg(["mean", "median", "count"])

num_papers                  degree         \
                                        mean median  count      mean median   
main_country name_inferred_gender                                             
br           female                 1.082260    1.0   1009  3.933598    3.0   
             male                   1.175334    1.0   2173  2.896457    2.0   
gb           female                 1.199206    1.0   3273  6.340972    3.0   
             male                   1.300421    1.0   5459  5.454296    2.0   
it           female                 1.222749    1.0   1688  5.394550    3.0   
             male                   1.380688    1.0   2879  4.472734    2.0   
kr           female                 1.179245    1.0    106  3.094340    2.0   
             male                   1.342466    1.0    146  3.095890    2.0   
ru           female                 1.153318    1.0   1311  1.893974    1.0   
             male                   1.207038    1.0   1222  1.697218    1.0   
us           female                 1.210154    1.0  12310  8.365556    4.0   
             male                   1.298093    1.0  17518  6.335598    3.0   

                                         weighted_degree                
                                   count            mean median  count  
main_country name_inferred_gender                                       
br           female                 1009        4.051536    3.0   1009  
             male                   2173        3.043718    2.0   2173  
gb           female                 3273        6.628170    3.0   3273  
             male                   5459        5.703426    2.0   5459  
it           female                 1688        5.572275    3.0   1688  
             male                   2879        4.737409    3.0   2879  
kr           female                  106        3.198113    2.5    106  
             male                    146        3.226027    2.0    146  
ru           female                 1311        1.955759    1.0   1311  
             male                   1222        1.805237    1.0   1222  
us           female                12310        9.062307    4.0  12310  
             male                  17518        6.815047    3.0  17518

#### Overall (at this stage)

median productivity is more or less the same for female and male authors in the sample. the median author has one paper in 2024 regardless of gender. the main gender differences are in network positions.

female authors show higher average and median degree than male authors in Brazil, UK, Italy, States. Russia shows a small female advantage in mean degree, but both genders have a median degree of 1 (possible explanation: weakly connected network in this case, maybe will be different other years). 

South Korea: female and male authors have similar mean and median degree.

weighted degree has the same pattern as degree, that suggests the gender differences are caused by broader coauthorship rather than by repeated collaboration. 

male authors have higher mean numbers of papers in every country.



### Gender composition of edges

In [101]:
edge_gender_df = edges_df.copy()

gender_smth = nodes_df[["author_id", "name_inferred_gender"]].copy()
gender_smth = gender_smth.rename(columns={"name_inferred_gender": "gender"})

In [102]:
edge_gender_df = edge_gender_df.merge(gender_smth.rename(columns={
    "author_id": "author_1_id", "gender": "gender_1"}), 
                                      on="author_1_id", how="left")

In [104]:
edge_gender_df = edge_gender_df.merge(gender_smth.rename(columns={
    "author_id": "author_2_id", "gender": "gender_2"}),
                                      on="author_2_id", how="left")

In [107]:
edge_gender_df[["author_1_id", "gender_1", "author_2_id", "gender_2", "weight"]].head()

,author_1_id,gender_1,author_2_id,gender_2,weight
0,https://openalex.org/A5000000076,unknown,https://openalex.org/A5063275369,unknown,1
1,https://openalex.org/A5000005418,female,https://openalex.org/A5019570644,male,1
2,https://openalex.org/A5000005418,female,https://openalex.org/A5029958956,male,1
3,https://openalex.org/A5000007462,female,https://openalex.org/A5048869928,female,1
4,https://openalex.org/A5000007462,female,https://openalex.org/A5087366708,female,1


In [109]:
def edge_gender(g1, g2):
    if pd.isna(g1) or pd.isna(g2):
        return "unknown"
        
    if g1 == "unknown" or g2 == "unknown":
        return "unknown"
        
    if g1 == "female" and g2 == "female":
        return "female-female"
    if g1 == "male" and g2 == "male":
        return "male-male"
    
    if {g1, g2} == {"female", "male"}:
        return "female-male"
    
    return "unknown"


edge_gender_df["edge_gender_type"] = edge_gender_df.apply(lambda row: edge_gender(row["gender_1"], row["gender_2"]),axis=1)

In [114]:
edge_gender_counts = edge_gender_df["edge_gender_type"].value_counts().rename_axis("edge_gender_type").reset_index(name="num_edges")
edge_gender_counts

,edge_gender_type,num_edges
0,female-male,63002
1,male-male,44697
2,unknown,43290
3,female-female,34307


In [112]:
c_edge_gender_df = edge_gender_df[edge_gender_df["edge_gender_type"] != "unknown"].copy()

In [115]:
c_edge_stats = (c_edge_gender_df.groupby("edge_gender_type", as_index=False)
                .agg(num_edges=("weight", "size"),total_weight=("weight", "sum")))

c_edge_stats["share_edges"] = c_edge_stats["num_edges"] / c_edge_stats["num_edges"].sum()
c_edge_stats["share_weight"] = c_edge_stats["total_weight"] / c_edge_stats["total_weight"].sum()

c_edge_stats = c_edge_stats.sort_values("num_edges", ascending=False)
c_edge_stats

,edge_gender_type,num_edges,total_weight,share_edges,share_weight
1,female-male,63002,67350,0.443657,0.442786
2,male-male,44697,47806,0.314754,0.314296
0,female-female,34307,36949,0.241588,0.242918


In [117]:
edge_gender_df.to_csv("data/edges_2024_with_gender.csv", index=False)
c_edge_stats.to_csv("data/edge_gender_stats_2024_without_NaN.csv", index=False)

so, we had male = 32046 and female = 21762. that means that probability that an author is female is approx. 0.404 and male approx. 0.596.

if the choice of co-authors was random, the probability of female-female pair would be p_f^2 = 0.163 which is much smaller than 0.241588.

the probability of male-male would be p_m^2 = 0.355 which is bigger than 0.314754.

the probability of female-male would be 2p_f*p_m = 0.482 which is bigger than 0.443657.

### By country

In [118]:
edge_country_df = c_edge_gender_df.copy()
country_lookup = nodes_df[["author_id", "main_country"]].copy()

In [121]:
edge_country_df = edge_country_df.merge(country_lookup.rename(columns={"author_id": "author_1_id", "main_country": "country_1"}),
                                        on="author_1_id",how="left")

In [122]:
edge_country_df = edge_country_df.merge(country_lookup.rename(columns={"author_id": "author_2_id", "main_country": "country_2"}),
                                        on="author_2_id",how="left")

In [123]:
within_country_edges = edge_country_df[edge_country_df["country_1"] == edge_country_df["country_2"]].copy()
within_country_edges["country"] = within_country_edges["country_1"]

In [126]:
country_edge_stats = (within_country_edges.groupby(["country", "edge_gender_type"], as_index=False)
                             .agg(num_edges=("weight", "size"),total_weight=("weight", "sum")))

In [132]:
country_edge_stats["share_edges"] = (country_edge_stats["num_edges"] / country_edge_stats.groupby("country")["num_edges"].transform("sum"))

country_edge_stats["share_weight"] = (country_edge_stats["total_weight"] / country_edge_stats.groupby("country")["total_weight"].transform("sum"))

country_edge_stats = country_edge_stats.sort_values(["country", "num_edges"], ascending=[True, False])

country_edge_table = country_edge_stats.pivot(index="country", columns="edge_gender_type",values="share_edges").fillna(0)

country_edge_table

edge_gender_type,female-female,female-male,male-male
country,,,
br,0.226936,0.408738,0.364326
gb,0.198577,0.450893,0.350530
it,0.198499,0.434183,0.367318
kr,0.287356,0.406130,0.306513
ru,0.324803,0.437500,0.237697
us,0.262160,0.446474,0.291366


### gender assortativity

In [154]:
gender_attr = nodes_df.set_index("author_id")["name_inferred_gender"].to_dict()
country_attr = nodes_df.set_index("author_id")["main_country"].to_dict()

nx.set_node_attributes(g, gender_attr, "gender")
nx.set_node_attributes(g, country_attr, "country")

In [155]:
classified_nodes = [n for n, d in g.nodes(data=True)
                    if d.get("gender") in ["female", "male"]]

g_classified = g.subgraph(classified_nodes).copy()

In [158]:
overall_gender_assortativity = nx.attribute_assortativity_coefficient(g_classified, "gender")
overall_gender_assortativity

0.10783768626702712

In [164]:
country_assort_rows = []

for country in sorted(nodes_df["main_country"].dropna().unique()):
    country_nodes = [n for n, d in g_classified.nodes(data=True)
                     if d.get("country") == country]
    
    subg = g_classified.subgraph(country_nodes).copy()
    
    if subg.number_of_edges() == 0:
        assort = float("nan")
    else:
        assort = nx.attribute_assortativity_coefficient(subg, "gender")
    
    country_assort_rows.append({"country": country,
                                "num_nodes": subg.number_of_nodes(),
                                "num_edges": subg.number_of_edges(),
                                "gender_assortativity": assort})

country_assort_df = pd.DataFrame(country_assort_rows)

In [162]:
country_assort_df

,country,num_nodes,num_edges,gender_assortativity
0,br,3695,5535,0.166666
1,gb,6947,19467,0.076740
2,it,3980,9023,0.104944
3,kr,576,261,0.187441
4,ru,2286,2031,0.118862
5,us,26709,95509,0.106275


the main is that all country networks show positive gender assortativity. there is same-gender clustering in co-authorship. 

the strongest values appear in South Korea and Brazil, while the weakest appears in the United Kingdom.

In [168]:
nodes_df.to_csv("data/nodes_2024_final.csv", index=False)
edges_df.to_csv("data/edges_2024.csv", index=False)
edge_gender_df.to_csv("data/edges_2024_with_gender.csv", index=False)
c_edge_stats.to_csv("data/edge_gender_stats_2024_classified.csv", index=False)
country_edge_stats.to_csv("data/country_edge_stats_2024.csv", index=False)
country_edge_table.to_csv("data/country_edge_share_table_2024.csv")
obs_exp_country.to_csv("data/obs_exp_country_2024.csv")
country_assort_df.to_csv("data/country_gender_assortativity_2024.csv", index=False)